In [1]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', *args, '-q'])

# hf_transfer: multi-threaded HF Hub downloads (10x faster, no stalling)
pip('install', 'hf_transfer')

# Unsloth: fastest QLoRA trainer (2x HF speed, 60% less VRAM)
pip('install', 'unsloth[colab-new]==2026.4.4', '--upgrade')

# Core training stack
pip('install',
    'trl>=0.8.6',
    'transformers>=4.51.0',
    'accelerate>=0.29',
    'peft>=0.10',
    'bitsandbytes>=0.43',
    'datasets>=2.18',
    'sentencepiece',
    'protobuf',
    'jsonschema',
    "huggingface-hub>=0.36.0,<1.0.0"
)

print("Packages installed")

Packages installed


In [2]:
# simple paths and optional Hugging Face login
from pathlib import Path
import json
import os

from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')

from huggingface_hub import login

DATA_DIR = Path("/content/drive/MyDrive/wos_data")
OUTPUT_DIR = DATA_DIR / "outputs"
REPORTS_DIR = DATA_DIR / "reports"
for path in (DATA_DIR, OUTPUT_DIR, REPORTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Train + validation only (see prepare_orchestration_dataset.py build in repo).
DATASET_PATH = DATA_DIR / "qwen3_orchestrator_train_val.jsonl"
# Held-out intents: same schema, never seen during training — evaluate after fit.
BLIND_TEST_PATH = DATA_DIR / "qwen3_orchestrator_blind_test.jsonl"
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Expected train/val JSONL at {DATASET_PATH}. Upload train_val + blind_test from prepare_orchestration_dataset.py build."
    )

HF_MODEL_REPO = "wcc0/wos_orch_qwen"
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    HF_TOKEN = ""

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)

print(json.dumps({
    "dataset_path": str(DATASET_PATH),
    "blind_test_path": str(BLIND_TEST_PATH),
    "blind_test_exists": BLIND_TEST_PATH.exists(),
    "output_dir": str(OUTPUT_DIR),
    "hf_model_repo": HF_MODEL_REPO,
    "hf_token_loaded": bool(HF_TOKEN),
}, indent=2))


Mounted at /content/drive
{
  "dataset_path": "/content/drive/MyDrive/wos_data/qwen3_orchestrator_dataset.cleaned.jsonl",
  "output_dir": "/content/drive/MyDrive/wos_data/outputs",
  "hf_model_repo": "wcc0/wos_orch_qwen",
  "hf_token_loaded": true
}


In [3]:
# load the Qwen3 32B model and attach LoRA adapters
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-32B-bnb-4bit"
ABSOLUTE_MAX_SEQ_LENGTH = 8192
LOAD_IN_4BIT = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.0

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=ABSOLUTE_MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|PAD_TOKEN|>"
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(json.dumps({
    "model_name": MODEL_NAME,
    "load_in_4bit": LOAD_IN_4BIT,
    "absolute_max_seq_length": ABSOLUTE_MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
}, indent=2))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-32B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.4 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


{
  "model_name": "unsloth/Qwen3-32B-bnb-4bit",
  "load_in_4bit": true,
  "absolute_max_seq_length": 8192,
  "lora_r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.0
}


In [4]:
# load the cleaned dataset, run a small sanity check, and render chat text for training
import hashlib
import json
import math
from datasets import Dataset

PLANNING_PREFIXES = (
    "i need to",
    "i'll",
    "i will",
    "let me",
    "first, i",
    "first i",
    "now i need to",
    "the user wants",
    "we need to",
)
PLANNING_SUBSTRINGS = (
    "i need to ask the user",
    "i need to clarify",
    "need clarification",
    "must ask the user",
    "must clarify",
    "before i can",
    "to proceed, i need",
    "i'll ask the user",
    "i will ask the user",
)

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def stable_bucket(row: dict) -> float:
    row_id = str(row.get("id") or json.dumps(row, sort_keys=True))
    digest = hashlib.sha1(row_id.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) / 0xFFFFFFFF

def split_name(row: dict) -> str:
    metadata = row.get("metadata") or {}
    return str(metadata.get("split") or "").strip().lower()

def looks_like_unresolved_planning(text: str) -> bool:
    normalized = " ".join(text.strip().lower().split())
    if not normalized:
        return False
    return normalized.startswith(PLANNING_PREFIXES) or any(
        token in normalized for token in PLANNING_SUBSTRINGS
    )

def semantic_stats(rows: list[dict]) -> dict:
    unresolved_planning = 0
    reasoning_equals_content = 0
    for row in rows:
        final_msg = row["messages"][-1]
        content = (final_msg.get("content") or "").strip()
        reasoning = (final_msg.get("reasoning_content") or "").strip()
        if reasoning and content and reasoning == content:
            reasoning_equals_content += 1
        if looks_like_unresolved_planning(content):
            unresolved_planning += 1
    return {
        "unresolved_planning": unresolved_planning,
        "final_reasoning_equals_content": reasoning_equals_content,
    }

def render_record(row: dict) -> dict:
    text = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    token_count = len(
        tokenizer.apply_chat_template(
            row["messages"],
            tokenize=True,
            add_generation_prompt=False,
        )
    )
    return {"text": text, "token_count": token_count}

records = load_jsonl(DATASET_PATH)
blind_records = load_jsonl(BLIND_TEST_PATH) if BLIND_TEST_PATH.exists() else []

if any(split_name(row) == "blind_test" for row in records):
    raise RuntimeError(
        "Training file contains blind_test rows — use qwen3_orchestrator_train_val.jsonl from prepare_orchestration_dataset.py build."
    )

sanity = semantic_stats(records + blind_records)
if sanity["unresolved_planning"] or sanity["final_reasoning_equals_content"]:
    raise RuntimeError(f"Dataset failed sanity checks: {json.dumps(sanity)}")

train_records = [row for row in records if split_name(row) == "train"]
# Validation only — never merge eval + blind (true blind generalization).
val_records = [row for row in records if split_name(row) in {"validation", "val", "eval"}]

if not train_records:
    raise RuntimeError("No train rows found in dataset metadata.")
if not val_records:
    val_records = [row for row in train_records if stable_bucket(row) < 0.02]
    train_records = [row for row in train_records if stable_bucket(row) >= 0.02]

train_rows = [render_record(row) for row in train_records]
val_rows = [render_record(row) for row in val_records]
blind_rows_raw = [render_record(row) for row in blind_records]

all_lengths = [row["token_count"] for row in train_rows + val_rows + blind_rows_raw]
observed_max = max(all_lengths)
MAX_SEQ_LENGTH = min(ABSOLUTE_MAX_SEQ_LENGTH, int(math.ceil(observed_max / 256.0) * 256))

train_rows = [row for row in train_rows if row["token_count"] <= MAX_SEQ_LENGTH]
val_rows = [row for row in val_rows if row["token_count"] <= MAX_SEQ_LENGTH]
blind_rows = [row for row in blind_rows_raw if row["token_count"] <= MAX_SEQ_LENGTH]

train_dataset = Dataset.from_list([{"text": row["text"]} for row in train_rows])
val_dataset = Dataset.from_list([{"text": row["text"]} for row in val_rows])
blind_dataset = (
    Dataset.from_list([{"text": row["text"]} for row in blind_rows])
    if blind_rows
    else None
)

length_stats = {
    "rows_train_val": len(records),
    "rows_blind": len(blind_records),
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "blind_examples": len(blind_rows) if blind_dataset is not None else 0,
    "blind_truncated": max(0, len(blind_rows_raw) - len(blind_rows)),
    "max_seq_length": MAX_SEQ_LENGTH,
    "observed_max_tokens": observed_max,
    "sanity": sanity,
}
(REPORTS_DIR / "length_stats.json").write_text(json.dumps(length_stats, indent=2), encoding="utf-8")
print(json.dumps(length_stats, indent=2))

{
  "rows": 1984,
  "train_examples": 1952,
  "validation_examples": 32,
  "max_seq_length": 4352,
  "observed_max_tokens": 4271,
  "sanity": {
    "unresolved_planning": 0,
    "final_reasoning_equals_content": 0
  }
}


In [7]:
# configure training and start when ready
from trl import SFTConfig, SFTTrainer

START_TRAINING_NOW = True
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 1e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR),
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        eos_token="<|im_end|>",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        weight_decay=0.01,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)

RUN_BLIND_EVAL_AFTER_TRAIN = True

if START_TRAINING_NOW:
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    if RUN_BLIND_EVAL_AFTER_TRAIN and blind_dataset is not None and len(blind_dataset) > 0:
        blind_metrics = trainer.evaluate(eval_dataset=blind_dataset)
        (REPORTS_DIR / "blind_eval_metrics.json").write_text(
            json.dumps(blind_metrics, indent=2),
            encoding="utf-8",
        )
        print(json.dumps({"blind_eval": blind_metrics}, indent=2))
    elif RUN_BLIND_EVAL_AFTER_TRAIN:
        print("Blind eval skipped: no blind_dataset rows (upload blind_test JSONL).")
else:
    print("Training is configured. Set START_TRAINING_NOW = True to begin.")

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/1952 [00:00<?, ? examples/s]

num_proc must be <= 32. Reducing num_proc to 32 for dataset of size 32.


Unsloth: Tokenizing ["text"] (num_proc=32):   0%|          | 0/32 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,952 | Num Epochs = 2 | Total steps = 488
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 268,435,456 of 33,030,558,720 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.380400,0.345462
100,0.331400,0.287261
150,0.295600,0.270512
200,0.292100,0.264424
250,0.240700,0.257842
300,0.234900,0.256665
350,0.244500,0.252187
400,0.228600,0.247790
450,0.217700,0.245310


In [9]:
# optional merged export for faster inference
SAVE_MERGED_16BIT = True
MERGED_DIR = DATA_DIR / "merged-16bit"

if SAVE_MERGED_16BIT:
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print(f"Saved merged model to {MERGED_DIR}")
else:
    print("Merged export is disabled. Set SAVE_MERGED_16BIT = True after training if you want a faster inference artifact.")

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00014.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00014-of-00014.safetensors:   0%|          | 0.00/2.08G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 14/14 [03:58<00:00, 17.04s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 14/14 [04:18<00:00, 18.45s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/wos_data/merged-16bit`
Saved merged model to /content/drive/MyDrive/wos_data/merged-16bit


In [8]:
# optional Hub push
PUSH_TO_HUB_NOW = True

if PUSH_TO_HUB_NOW:
    if not HF_TOKEN:
        raise ValueError("Set HF_TOKEN in the Colab environment before pushing.")
    model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
    print(f"Pushed model artifacts to {HF_MODEL_REPO}")
else:
    print("Hub push is disabled. Set PUSH_TO_HUB_NOW = True after training if you want to upload the result.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 13.8kB / 1.07GB            

Saved model to https://huggingface.co/wcc0/wos_orch_qwen


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mprqfmh3av/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed model artifacts to wcc0/wos_orch_qwen
